In [ ]:
import numpy as np
import sys
from pathlib import Path
import pandas as pd

# Auto-detect project root and add to path
# (Can override with PROJECT_ROOT environment variable or .project_root file)
cwd = Path.cwd()
project_root = None
# Check current directory and parents for train.py
current = cwd
for _ in range(5):
    if (current / "train.py").exists():
        project_root = current
        break
    current = current.parent

if project_root and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.path_utils import setup_path
setup_path()

import os
import importlib

# Point MGT_PAPER_RESULTS_ROOT at the run's results tree (the directory holding
# post-processed-results/) before starting the kernel, e.g. in project_settings.
# It defaults to results/paper_run under the checkout.

import analysis.model_testing.dataset_processing_config as _dataset_config
importlib.reload(_dataset_config)
import analysis.talented_models_v3 as _talented_models
importlib.reload(_talented_models)
import analysis.model_testing.zero_cost_metric_analysis as _zero_cost_analysis
importlib.reload(_zero_cost_analysis)

from analysis.talented_models_v3 import ZC_METRICS, ORDERED_DATASETS, OUT_DIR
from analysis.model_testing.dataset_processing_config import DATASET_CONFIGS
OUT_DIR = OUT_DIR / "may_2026_rerun"
from analysis.model_testing.zero_cost_metric_analysis import corr_for_dataset, ZERO_COST_METRICS, ID_REGEX, _load_and_align

_probe = Path(DATASET_CONFIGS["motionsense"]["full_results_pkl"])
if not _probe.exists():
    raise FileNotFoundError(
        f"{_probe} is missing. Set MGT_PAPER_RESULTS_ROOT to the May 2026 results directory before starting the kernel."
    )
print("using results from", _probe.parent.parent)


### plot params 
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
PALETTE10 = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#8c564b",  # brown
    "#e377c2",  # pink
    "#7f7f7f",  # gray
    "#bcbd22",  # olive
    "#17becf",  # cyan
]
plt.rcParams.update({
    "font.size": 14,        # base size for everything
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "figure.titlesize": 14,
})

MODEL_TYPE_FILTER = None


def _save_fig_png_pdf(fig, outpath):
    outpath = Path(outpath)
    outpath.parent.mkdir(parents=True, exist_ok=True)
    stem = outpath.stem
    if MODEL_TYPE_FILTER is not None:
        stem = f"{stem}_model_filter={MODEL_TYPE_FILTER}"
    base = outpath.with_name(stem)
    for suf in (".pdf", ".png"):
        p = base.with_suffix(suf)
        fig.savefig(p, dpi=300, bbox_inches="tight")
        print(f"[plot saved → {p}]")


In [ ]:
import os
import pandas as pd

for dataset in ['motionsense', 'hhar', 'rwhar', 'mobiactv2', 'myogym', 'pamap2']:
    mdf = pd.read_pickle(DATASET_CONFIGS[dataset]['full_results_pkl'])
    if MODEL_TYPE_FILTER is not None:
        mdf = mdf[mdf.model_type == MODEL_TYPE_FILTER]
    print(dataset)
    print(len(mdf))
    print(len(mdf[['best_test_f1_macro']].dropna()))
    print(mdf['model_type'].value_counts())

In [ ]:
K_PERCENTAGE = 0.1  # fraction of pool size → k (override as needed)
K_TO_USE = int(round(len(mdf) * K_PERCENTAGE))
NUM_MODELS = len(mdf)
print('using k_percentage', K_PERCENTAGE, '→ k', K_TO_USE, 'and num models', NUM_MODELS)

In [ ]:
def extract_model_id_from_run_dir(run_dir_series):
    """Extract model_id from run_dir, supporting multiple formats."""
    # Try arch_seed,combo_idx format first
    model_id = run_dir_series.str.extract(r"(arch_seed=\d+,combo_idx=\d+)")
    if model_id[0].isna().all():
        # Try arch_seed,init_name,init_seed format
        model_id = run_dir_series.str.extract(r"(arch_seed=\d+,init_name=\w+,init_seed=\d+)")
    if model_id[0].isna().all():
        # Fallback to train_tune trial ID
        model_id = run_dir_series.str.extract(r"(train_tune_\w+_\d+_\d+)")
    return model_id[0]

def compute_talent_rate_and_delta(dataset, k, use_abs_zcp: bool = True, *, model_type=None, model_types=None):
    deltas = {}
    talent_rates = {}
    expected_deltas = {}
    initial_metrics = ["initial_val_f1_macro"] + ZC_METRICS + ["ensemble"]
    zcp = _load_and_align(cfg=DATASET_CONFIGS[dataset], id_regex=ID_REGEX, keep_run_dir=True)
    fdf = pd.read_pickle(DATASET_CONFIGS[dataset]["full_results_pkl"]).drop('initial_val_f1_macro', axis=1, errors='ignore')
    # Extract model_id from run_dir (supports multiple formats)
    fdf["model_id"] = extract_model_id_from_run_dir(fdf["run_dir"])
    zcp = zcp.merge(fdf[['run_dir', 'best_val_f1_macro', 'model_id', 'model_type']], on='run_dir')
    if model_type is not None:
        zcp = zcp[zcp["model_type"] == model_type]
    if model_types is not None:
        zcp = zcp[zcp["model_type"].isin(model_types)]
    if use_abs_zcp:
        for m in ZC_METRICS:
           zcp[m] = zcp[m].abs()
    zcp["ensemble"] = zcp[ZC_METRICS].rank(pct=True).mean(axis=1)

    deltas[dataset] = {}
    talent_rates[dataset] = {}
    expected_deltas[dataset] = {}
    top_models_by_val= zcp.sort_values(by="best_val_f1_macro", ascending=False).head(k)
    top_test_score_by_val = top_models_by_val['best_test_f1_macro'].iloc[0]
    for m in initial_metrics:
        # first order by zcp and get top k
        top_by_zcp = zcp.sort_values(by=m, ascending=False).head(k)
        # top_by_zcp['_sum'] = top_by_zcp[m] + top_by_zcp['best_val_f1_macro']
        # now resort the subset by val
        top_zcp_by_val = top_by_zcp.sort_values(by="best_val_f1_macro", ascending=False).reset_index(drop=True)
        # deltas[dataset][m] = top_test_score_by_val - top_zcp_by_val.head(max(1, smoothing_factor))['best_test_f1_macro'].max()
        deltas[dataset][m] = top_test_score_by_val - top_zcp_by_val['best_test_f1_macro'].iloc[0]
        top_predicted_models = set(top_by_zcp.model_id.unique())
        top_actual_models = set(top_models_by_val.model_id.unique()) 
        talent_rates[dataset][m] = len(top_predicted_models & top_actual_models) / len(top_actual_models)

    # expected deltas

    davg = []
    talent_rate = []
    top_models = set(top_models_by_val.model_id.unique()) 
    for i in range(1000):
        sampled = zcp.sample(n=k, replace=False)
        d = top_test_score_by_val - sampled.sort_values(by='best_val_f1_macro', ascending=False)['best_test_f1_macro'].iloc[0]
        davg.append(d)
        tr = len(set(sampled.model_id.unique()) & top_models) / len(top_models)
        talent_rate.append(tr)
    dstdv = np.std(davg)
    davg = np.mean(davg)
    trstdev = np.std(talent_rate)
    travg = np.mean(talent_rate)
    expected_vals = {dataset: {'expected_talent_rate': travg * 100, 'expected_delta': davg, 'expected_delta_var': dstdv ** 2, 'expected_talent_rate_var': trstdev}}
    return talent_rates, deltas, expected_vals

def compute_talent_stats(use_abs_zcp, datasets: list[str], k=75, *, model_type=None, model_types=None):
    deltas = []
    talent_rates = []
    expected_vals = []
    for dataset in datasets:
        trs, ds, expvals = compute_talent_rate_and_delta(
            dataset,
            k=k,
            use_abs_zcp=use_abs_zcp,
            model_type=model_type,
            model_types=model_types,
        )
        deltas.append(pd.DataFrame(ds).T.reset_index())
        talent_rates.append(pd.DataFrame(trs).T.reset_index())
        expected_vals.append(pd.DataFrame(expvals).T.reset_index())
    delta_df = pd.concat(deltas).rename(columns={'index': 'dataset'})
    talent_rate_df = pd.concat(talent_rates).rename(columns={'index': 'dataset'})
    expvals_df = pd.concat(expected_vals).rename(columns={'index': 'dataset'})
    delta_df = delta_df.reset_index(drop=True)
    talent_rate_df = talent_rate_df.reset_index(drop=True)
    expvals_df = expvals_df.reset_index(drop=True)

    metric_cols = [c for c in talent_rate_df.columns if c != "dataset"]

    delta_df = (
        delta_df
        .melt(id_vars="dataset", value_vars=metric_cols,
            var_name="metric", value_name="delta")
        .dropna(subset=["delta"])
    ).sort_values(by='dataset')
    # delta_df['delta'] *= 100

    # Wide -> long
    talent_rate_df = (
        talent_rate_df
        .melt(id_vars="dataset", value_vars=metric_cols,
            var_name="metric", value_name="talent")
        .dropna(subset=["talent"])
    ).sort_values(by='dataset')

    talent_rate_df['talent'] *= 100
    talent_rate_df = talent_rate_df.merge(delta_df[['dataset', 'metric', 'delta']], on=['dataset', 'metric']).sort_values(by='dataset')
    talent_rate_df.head()
    return talent_rate_df, delta_df, expvals_df

In [ ]:
k = 10
talent_rate_df, delta_df, expvals_df = compute_talent_stats(use_abs_zcp=False, datasets=ORDERED_DATASETS, k=k, model_type=MODEL_TYPE_FILTER)
talent_rate_df_k1, delta_df_k1, expvals_df_k1 = compute_talent_stats(use_abs_zcp=False, datasets=ORDERED_DATASETS, k=1, model_type=MODEL_TYPE_FILTER)
delta_df_k1['k']=1
delta_df['k']=k
expvals_df["k"] = k
expvals_df_k1['k'] = 1
# talent_rate_df_abs, delta_df_abs, expvals_df = compute_talent_stats(use_abs_zcp=True, datasets=ORDERED_DATASETS, k=10)
# IMBALANCE_DATASETS = ["motionsense", "motionsense_.1", "motionsense_.01", "motionsense_.001", "rwhar", "rwhar_.1", "rwhar_.01", "rwhar_.001"]
# talent_rate_df_imb, delta_df_imb, expvals_df_imb = compute_talent_stats(use_abs_zcp=False, datasets=IMBALANCE_DATASETS)
# talent_rate_df_abs, delta_df_abs, expvals_df = compute_talent_stats(use_abs_zcp=True)

In [ ]:
(
    talent_rate_df_large_k, 
    delta_df_large_k, 
    expvals_df_large_k
) = compute_talent_stats(
    use_abs_zcp=False, 
    datasets=ORDERED_DATASETS, 
    k=K_TO_USE, 
    model_type=MODEL_TYPE_FILTER
)

In [ ]:
delta_only_df = pd.concat([delta_df, delta_df_k1])
exp_vals_delta_only_df = pd.concat([expvals_df, expvals_df_k1])
exp_vals_delta_only_df

In [ ]:
def plot_delta_two_panels(
    df,                               # long df with: dataset, metric, delta, k
    outpath,                          # Path or str to save (single figure with 2 panels)
    *,
    k_top=1,
    k_bottom=None,                    # if None -> max(df['k'])
    dataset_order=None,               # list[str]; if None -> sorted unique in df
    expected_df=None,                 # DataFrame: dataset, expected_delta, expected_delta_var [, k]
    figsize=(18, 10),
    legend_ncols=5,
    legend_loc="lower center",
    zero_label_tol=1e-5,
    rs_label="Random search",
    rs_color="#66CCEE",
    rs_ecolor="#000000",
    rs_err_capsize=4,
):
    def _norm_ds(ds: str) -> str:
        ds = ds.lower()
        return "mobiact" if ds == "mobiactv2" else ds

    def _exp_for_k(exp_df, k_val):
        if exp_df is None:
            return {}
        e = exp_df.copy()
        if "k" in e.columns:
            e = e[e["k"] == k_val]
        out = {}
        for _, r in e.iterrows():
            ds = _norm_ds(str(r["dataset"]))
            mu = float(r["expected_delta"])
            std = float(np.sqrt(r["expected_delta_var"])) if "expected_delta_var" in e.columns else np.nan
            out[ds] = (mu, std)
        return out

    if dataset_order is None:
        dataset_order = sorted(df["dataset"].unique(), key=str)
    if k_bottom is None:
        k_bottom = int(df["k"].max())

    top_df = df[df["k"] == k_top].copy()
    bot_df = df[df["k"] == k_bottom].copy()
    exp_top = _exp_for_k(expected_df, k_top)
    exp_bot = _exp_for_k(expected_df, k_bottom)

    plt.style.use("default")
    plt.rcParams.update({
        "font.size": 12,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 14,
        "legend.fontsize": 12,
    })

    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=figsize, sharex=True)
    palette = PALETTE10

    def _style_axes(ax, border: bool):
        ax.grid(True, axis="y", alpha=0.25)
        ax.set_axisbelow(True)

        if border:
            for sp in ax.spines.values():
                sp.set_visible(True)
                sp.set_linewidth(1.0)
                sp.set_edgecolor("black")
            ax.tick_params(axis="both", length=3)
        else:
            # No border around the plot
            for sp in ax.spines.values():
                sp.set_visible(True)
            # Optional: remove tick marks but keep labels
            ax.tick_params(axis="both", length=0)

    def _draw_panel(ax, dfi, exp_map, k_label):
        metrics = sorted(dfi["metric"].unique())
        n_groups = len(metrics) + 1     # +1 slot for Random search
        x = np.arange(len(dataset_order))
        width = 0.8 / max(n_groups, 1)

        max_val, min_val = -np.inf, np.inf

        # regular metric bars (now with black outlines)
        for j, m in enumerate(metrics):
            offsets = x + (j - n_groups / 2) * width + width / 2
            sub = dfi[dfi["metric"] == m].set_index("dataset")
            vals = [sub.loc[ds, "delta"] if ds in sub.index else np.nan for ds in dataset_order]

            bars = ax.bar(
                offsets, vals, width=width, label=m,
                color=palette[j % len(palette)],
                edgecolor="black", linewidth=0.9
            )

            for rect, val in zip(bars, vals):
                if np.isfinite(val) and abs(val) <= zero_label_tol:
                    ax.annotate(
                        "0",
                        (rect.get_x() + rect.get_width()/2, rect.get_height()),
                        textcoords="offset points", xytext=(0, 6),
                        ha="center", va="bottom", fontsize=9
                    )

            fin = [v for v in vals if np.isfinite(v)]
            if fin:
                max_val = max(max_val, max(fin))
                min_val = min(min_val, min(fin))

        # random search bars (mean with SD error bars) — outlined
        j_rs = len(metrics)
        rs_offsets = x + (j_rs - n_groups / 2) * width + width / 2
        rs_vals, rs_yerr = [], []
        for ds in dataset_order:
            key = _norm_ds(ds)
            if key in exp_map:
                mu, std = exp_map[key]
                rs_vals.append(mu)
                rs_yerr.append(std if np.isfinite(std) else 0.0)
                if np.isfinite(mu):
                    max_val = max(max_val, mu + (std if np.isfinite(std) else 0.0))
                    min_val = min(min_val, mu - (std if np.isfinite(std) else 0.0))
            else:
                rs_vals.append(np.nan)
                rs_yerr.append(0.0)

        ax.bar(
            rs_offsets, rs_vals, width=width, label=rs_label,
            color=rs_color, edgecolor="black", linewidth=0.9,
            yerr=rs_yerr,
            error_kw=dict(elinewidth=1.4, ecolor=rs_ecolor, capsize=rs_err_capsize, capthick=1.4)
        )

        # ticks / labels
        tick_labels = [("mobiact" if d == "mobiactv2" else d) for d in dataset_order]
        tick_labels = [d.upper() if d.lower() not in {"motionsense", "mobiact", "myogym"}
                       else d.capitalize() for d in tick_labels]
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels, rotation=0)
        ax.set_ylabel("Delta from Best Model")

        # annotate k (no titles)
        ax.text(0.01, 0.98, f"k = {k_label}", transform=ax.transAxes,
                ha="left", va="top", fontsize=14)

        # y-lims with headroom
        if np.isfinite(max_val) and np.isfinite(min_val):
            pad = 0.1 * (max_val - min_val if max_val > min_val else 1.0)
            ax.set_ylim(min_val - 0.2 * pad, max_val + pad)

        _style_axes(ax, border=False)
        return ax.get_legend_handles_labels()

    h1, l1 = _draw_panel(ax_top, top_df, exp_top, k_top)
    h2, l2 = _draw_panel(ax_bot, bot_df, exp_bot, k_bottom if k_bottom is not None else "N/A")

    # shared legend (dedupe)
    handles_all = h1 + h2
    labels_all  = l1 + l2
    seen, H, L = set(), [], []
    for h, l in zip(handles_all, labels_all):
        if l and l not in seen:
            seen.add(l); H.append(h); L.append(l)

    # spacing & legend placement
    ax_top.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    fig.subplots_adjust(hspace=0.05, bottom=0.18)
    fig.legend(
        H, L,
        loc="upper center",
        bbox_to_anchor=(0.5, -.15),
        bbox_transform=ax_bot.transAxes,
        ncol=min(len(L), legend_ncols),
        frameon=True, edgecolor="black", facecolor="white"
    )

    _save_fig_png_pdf(fig, outpath)
    plt.show()

plot_delta_two_panels(
    df=delta_only_df,                   # dataset, metric, delta, k
    outpath=OUT_DIR / "delta_k1_and_kx.pdf",
    k_top=1,
    k_bottom=10,
    dataset_order=ORDERED_DATASETS,
    expected_df=exp_vals_delta_only_df, # must include expected_delta & expected_delta_var
    rs_label="Random search"
)

In [ ]:
# show spearman correlations for zero-cost proxies
def run_all_correlations(
    use_abs: bool = False, min_cluster_size: int = 10
) -> dict[str, dict[str, pd.DataFrame]]:
    """Run correlations for every dataset in dataset_configs and return nested dict."""
    results = {}
    for name, cfg in DATASET_CONFIGS.items():
        try:
            results[name] = corr_for_dataset(
                name,
                cfg,
                use_abs=use_abs,
                max_cluster_size=min_cluster_size,
            )
            print(f"{name}: done")
        except Exception as exc:
            print(f"Failed to run dataset {name}: {exc}")
    return results

corrs = run_all_correlations()
records_abs = []
for ds, corr_dict in corrs.items():
    spearman = corr_dict["spearman"]
    for met in ZERO_COST_METRICS + ["ensemble", "initial_val_f1_macro"]:
        if met in spearman.index and "best_test_f1_macro" in spearman.columns:
            records_abs.append({
                "dataset":     ds,
                "metric":      met,
                "correlation": float(spearman.loc[met, "best_test_f1_macro"])
            })
df = pd.DataFrame(records_abs)
zcp_trd_df_spearman = df[df.dataset.isin(ORDERED_DATASETS)]
zcp_trd_df_spearman.head()

corrs_abs = run_all_correlations(use_abs=True)
records_abs = []
for ds, corr_dict_abs in corrs_abs.items():
    spearman_abs = corr_dict_abs["spearman"]
    for met in ZERO_COST_METRICS + ["ensemble", "initial_val_f1_macro"]:
        if met in spearman_abs.index and "best_test_f1_macro" in spearman_abs.columns:
            records_abs.append({
                "dataset":     ds,
                "metric":      met,
                "correlation": float(spearman_abs.loc[met, "best_test_f1_macro"])
            })
df_abs =  pd.DataFrame(records_abs)
zcp_trd_df_spearman_abs = df_abs[df_abs.dataset.isin(ORDERED_DATASETS)]
zcp_trd_df_spearman.head()

In [ ]:
def plot_spearman_from_df_single(
    df,                 # columns: dataset, metric, correlation
    outpath,
    figsize=(18, 5),
    dataset_order: list[str] | None = None,
):
    plt.style.use("ggplot")
    plt.rcParams.update({
        "font.size": 12,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 14,
        "legend.fontsize": 12,
    })

    # dataset order + label normalization
    def _norm_label(d: str) -> str:
        d = "mobiact" if d == "mobiactv2" else d
        return d.capitalize() if d.lower() in {"motionsense", "mobiact", "myogym"} else d.upper()

    datasets = dataset_order or list(df["dataset"].unique())
    metrics = sorted(df["metric"].unique())
    n_metrics = max(len(metrics), 1)

    x = np.arange(len(datasets))
    width = 0.8 / n_metrics  # group width = 0.8

    fig, ax = plt.subplots(figsize=figsize, dpi=200)
    fig.patch.set_facecolor("white")
    ax.set_facecolor(plt.rcParams["axes.facecolor"])
    ax.grid(True, axis="y", color="white", linewidth=1.2)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("black")
        ax.spines[side].set_linewidth(0.9)
    ax.tick_params(axis="both", length=0)

    # grouped bars by metric
    for j, m in enumerate(metrics):
        offsets = x + (j - n_metrics / 2) * width + width / 2
        sub = df[df["metric"] == m].set_index("dataset")
        vals = [float(sub.loc[ds, "correlation"]) if ds in sub.index else np.nan for ds in datasets]
        ax.bar(
            offsets, vals, width=width, label=m,
            color=PALETTE10[j % len(PALETTE10)],
            edgecolor="black", linewidth=0.9
        )

    # y label & limits (Spearman ρ ∈ [-1, 1], with padding)
    ax.set_ylabel("Spearman ρ")
    yvals = df["correlation"].to_numpy(dtype=float)
    yfinite = yvals[np.isfinite(yvals)]
    ymin, ymax = -.4, 1.0
    pad = 0.08 * (ymax - ymin)
    ax.set_ylim(ymin , ymax)
    from matplotlib.ticker import MultipleLocator, FormatStrFormatter
    # ax.yaxis.set_major_locator(MultipleLocator(0.1))
    # ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f')) 

    # x ticks
    ax.set_xticks(x)
    ax.set_xticklabels([_norm_label(d) for d in datasets], rotation=0, ha="right")
    ax.margins(x=0.02)

    handles, labels = ax.get_legend_handles_labels()
    fig.subplots_adjust(bottom=0.24)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.04),
        ncol=min(len(labels), 5),
        frameon=True,
        edgecolor="black",
        facecolor="white",
    )

    # save & show
    _save_fig_png_pdf(fig, outpath)
    plt.show()
    
plot_spearman_from_df_single(
    df=zcp_trd_df_spearman,
    outpath=OUT_DIR / "zcp_spearman_single.pdf",
    dataset_order=ORDERED_DATASETS
)

In [ ]:
# Ensemble-only talent/delta — two bar layouts from the same `df`:
#   (1) x = model type, bars = per-dataset; (2) x = dataset, bars = per–model type.
def _ds_lbl(d: str) -> str:
    d = "mobiact" if d == "mobiactv2" else d
    return d.capitalize() if d.lower() in {"motionsense", "mobiact", "myogym"} else d.upper()

model_types = sorted({
    mt
    for ds in ORDERED_DATASETS
    for mt in pd.read_pickle(DATASET_CONFIGS[ds]["full_results_pkl"])["model_type"].dropna().unique()
})

k_percentage = K_PERCENTAGE  # fraction of each dataset’s model-type pool; same k across datasets (min across pools)

rows = []
for mt in model_types:
    per_ds_k = []
    for ds in ORDERED_DATASETS:
        c = int((pd.read_pickle(DATASET_CONFIGS[ds]["full_results_pkl"])["model_type"] == mt).sum())
        assert c > 0
        per_ds_k.append(int(round(c * k_percentage)))
    k_mt = min(per_ds_k) if per_ds_k else 1
    tr_df, _, _ = compute_talent_stats(use_abs_zcp=False, datasets=ORDERED_DATASETS, k=k_mt, model_type=mt)
    e = tr_df[tr_df["metric"] == "ensemble"][['dataset', 'talent', 'delta']].copy()
    e["model_type"] = mt
    rows.append(e)

df = pd.concat(rows, ignore_index=True)
x = np.arange(len(model_types))
w = 0.8 / len(ORDERED_DATASETS)

fig, (ax_t, ax_d) = plt.subplots(2, 1, figsize=(18, 8), sharex=True)
for j, ds in enumerate(ORDERED_DATASETS):
    off = x + (j - len(ORDERED_DATASETS) / 2) * w + w / 2
    sub = df[df["dataset"] == ds].set_index("model_type")
    tvals = [sub.loc[mt, "talent"] if mt in sub.index else np.nan for mt in model_types]
    dvals = [sub.loc[mt, "delta"] if mt in sub.index else np.nan for mt in model_types]
    c = PALETTE10[j % len(PALETTE10)]
    ax_t.bar(off, tvals, width=w, label=_ds_lbl(ds), color=c, edgecolor="black", linewidth=0.8)
    ax_d.bar(off, dvals, width=w, label=_ds_lbl(ds), color=c, edgecolor="black", linewidth=0.8)

ax_t.set_ylabel("Talent Rate")
ax_t.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax_t.grid(True, axis="y", alpha=0.25)
ax_t.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

ax_d.set_ylabel("Delta from Best Model")
ax_d.grid(True, axis="y", alpha=0.25)
ax_d.set_xticks(x)
ax_d.set_xticklabels(model_types, rotation=15, ha="right")

for ax in (ax_t, ax_d):
    for s in ("top", "right", "left", "bottom"):
        ax.spines[s].set_visible(True)
    ax.tick_params(axis="both", length=0)

fig.subplots_adjust(hspace=0.05, bottom=0.22)
fig.legend(loc="upper center", bbox_to_anchor=(0.5, -0.28), bbox_transform=ax_d.transAxes,
           ncol=min(6, len(ORDERED_DATASETS)), frameon=True, edgecolor="black", facecolor="white")

_save_fig_png_pdf(fig, OUT_DIR / "ensemble_talent_and_delta_by_model_type.pdf")
plt.show()

# Swapped: x grouped by dataset; bars = model type (legend = model type)
x_ds = np.arange(len(ORDERED_DATASETS))
w_mt = 0.8 / len(model_types)
fig2, (ax_t2, ax_d2) = plt.subplots(2, 1, figsize=(18, 8), sharex=True)
for j, mt in enumerate(model_types):
    off = x_ds + (j - len(model_types) / 2) * w_mt + w_mt / 2
    sub = df[df["model_type"] == mt].set_index("dataset")
    tvals = [sub.loc[ds, "talent"] if ds in sub.index else np.nan for ds in ORDERED_DATASETS]
    dvals = [sub.loc[ds, "delta"] if ds in sub.index else np.nan for ds in ORDERED_DATASETS]
    c = PALETTE10[j % len(PALETTE10)]
    ax_t2.bar(off, tvals, width=w_mt, label=mt, color=c, edgecolor="black", linewidth=0.8)
    ax_d2.bar(off, dvals, width=w_mt, label=mt, color=c, edgecolor="black", linewidth=0.8)

ax_t2.set_ylabel("Talent Rate")
ax_t2.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax_t2.grid(True, axis="y", alpha=0.25)
ax_t2.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

ax_d2.set_ylabel("Delta from Best Model")
ax_d2.grid(True, axis="y", alpha=0.25)
ax_d2.set_xticks(x_ds)
ax_d2.set_xticklabels([_ds_lbl(ds) for ds in ORDERED_DATASETS], rotation=15, ha="right")

for ax in (ax_t2, ax_d2):
    for s in ("top", "right", "left", "bottom"):
        ax.spines[s].set_visible(True)
    ax.tick_params(axis="both", length=0)

fig2.subplots_adjust(hspace=0.05, bottom=0.22)
fig2.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.28),
    bbox_transform=ax_d2.transAxes,
    ncol=min(6, len(model_types)),
    frameon=True,
    edgecolor="black",
    facecolor="white",
)

_save_fig_png_pdf(fig2, OUT_DIR / "ensemble_talent_and_delta_by_dataset.pdf")
plt.show()

In [ ]:
# Combined Spearman/delta/talent-rate figure with one shared legend.
# Uses the grey/white-grid style from the noise plot.
from matplotlib.ticker import PercentFormatter


def plot_combined_delta_talent_panels(
    *,
    delta_df_small_k,
    delta_df_mid_k,
    talent_df_large_k,
    delta_df_large_k,
    spearman_df=None,
    exp_df_small_k=None,
    exp_df_mid_k=None,
    exp_df_large_k=None,
    outpath=None,
    dataset_order=None,
    k_small=1,
    k_mid=20,
    k_large=None,
    include_spearman=True,
    figsize=(18, 16),  # height scaled to 75% below based on panel count
    legend_ncols=5,
    rs_label="Random search",
    rs_color="#66CCEE",
    rs_ecolor="#000000",
    rs_err_capsize=4,
):
    def _norm_ds(ds: str) -> str:
        ds = ds.lower()
        return "mobiact" if ds == "mobiactv2" else ds

    def _ds_lbl(ds: str) -> str:
        ds = _norm_ds(ds)
        return ds.capitalize() if ds in {"motionsense", "mobiact", "myogym"} else ds.upper()

    def _exp_map(exp_df, value_col, err_col, *, sqrt_err=False, err_scale=1.0):
        if exp_df is None:
            return {}
        out = {}
        for _, row in exp_df.iterrows():
            ds = _norm_ds(str(row["dataset"]))
            mu = float(row[value_col]) if value_col in row and pd.notna(row[value_col]) else np.nan
            err = float(row[err_col]) if err_col in row and pd.notna(row[err_col]) else np.nan
            if sqrt_err and np.isfinite(err):
                err = float(np.sqrt(err))
            if np.isfinite(err):
                err *= err_scale
            out[ds] = (mu, err)
        return out

    def _style_ax(ax):
        ax.set_facecolor(plt.rcParams["axes.facecolor"])
        ax.grid(True, axis="y", color="white", linewidth=1.2)
        ax.set_axisbelow(True)
        for side in ("top", "right", "left", "bottom"):
            ax.spines[side].set_visible(True)
            ax.spines[side].set_color("black")
            ax.spines[side].set_linewidth(0.9)
        ax.tick_params(axis="both", length=0)

    def _draw_panel(ax, df, value_col, y_label, panel_label, exp_map=None, percent_xmax=None):
        metrics = sorted(df["metric"].unique())
        n_groups = len(metrics) + (1 if exp_map else 0)
        x = np.arange(len(dataset_order))
        width = 0.8 / max(n_groups, 1)
        max_val, min_val = -np.inf, np.inf

        for j, metric in enumerate(metrics):
            offsets = x + (j - n_groups / 2) * width + width / 2
            sub = df[df["metric"] == metric].set_index("dataset")
            vals = [sub.loc[ds, value_col] if ds in sub.index else np.nan for ds in dataset_order]
            ax.bar(
                offsets,
                vals,
                width=width,
                label=metric,
                color=PALETTE10[j % len(PALETTE10)],
                edgecolor="black",
                linewidth=0.8,
            )
            finite_vals = [v for v in vals if np.isfinite(v)]
            if finite_vals:
                max_val = max(max_val, max(finite_vals))
                min_val = min(min_val, min(finite_vals))

        if exp_map:
            j = len(metrics)
            offsets = x + (j - n_groups / 2) * width + width / 2
            vals, yerr = [], []
            for ds in dataset_order:
                mu, err = exp_map.get(_norm_ds(ds), (np.nan, 0.0))
                vals.append(mu)
                yerr.append(err if np.isfinite(err) else 0.0)
                if np.isfinite(mu):
                    err_for_lim = err if np.isfinite(err) else 0.0
                    max_val = max(max_val, mu + err_for_lim)
                    min_val = min(min_val, mu - err_for_lim)

            ax.bar(
                offsets,
                vals,
                width=width,
                label=rs_label,
                color=rs_color,
                edgecolor="black",
                linewidth=0.8,
                yerr=yerr,
                error_kw=dict(
                    elinewidth=1.2,
                    ecolor=rs_ecolor,
                    capsize=rs_err_capsize,
                    capthick=1.2,
                ),
            )

        ax.set_ylabel(y_label)
        if percent_xmax is not None:
            ax.yaxis.set_major_formatter(PercentFormatter(xmax=percent_xmax, decimals=0))
        ax.text(0.01, 0.95, panel_label, transform=ax.transAxes, ha="left", va="top", fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels([_ds_lbl(ds) for ds in dataset_order], rotation=0)
        ax.margins(x=0.02)
        _style_ax(ax)

        if value_col == "correlation":
            ax.set_ylim(-0.4, 1.0)
        elif np.isfinite(max_val) and np.isfinite(min_val):
            span = max_val - min_val if max_val > min_val else 1.0
            pad = 0.12 * span
            ax.set_ylim(min_val - 0.2 * pad, max_val + pad)

        return ax.get_legend_handles_labels()

    if dataset_order is None:
        dataset_order = ORDERED_DATASETS
    if k_large is None:
        k_large = K_TO_USE

    plt.style.use("ggplot")
    show_spearman = include_spearman and spearman_df is not None
    n_panels = 5 if show_spearman else 4
    # Keep per-panel height at 75% of the original 5-panel figure (16 / 5)
    fig_w, fig_h = figsize
    fig_h = n_panels * (fig_h / 5) * 0.75
    fig, axes = plt.subplots(n_panels, 1, figsize=(fig_w, fig_h), sharex=True, dpi=200)
    fig.patch.set_facecolor("white")

    panel_specs = [
        (axes[0], delta_df_small_k, "delta", "\u0394 from Best Model", f"(A) k = {k_small}", _exp_map(exp_df_small_k, "expected_delta", "expected_delta_var", sqrt_err=True), 1.0),
        (axes[1], delta_df_mid_k, "delta", "\u0394 from Best Model", f"(B) k = {k_mid}", _exp_map(exp_df_mid_k, "expected_delta", "expected_delta_var", sqrt_err=True), 1.0),
        (axes[2], talent_df_large_k, "talent", "Talent Rate", f"(C) k = {k_large}", _exp_map(exp_df_large_k, "expected_talent_rate", "expected_talent_rate_var", err_scale=100.0), 100.0),
        (axes[3], delta_df_large_k, "delta", "\u0394 from Best Model", f"(D) k = {k_large}", _exp_map(exp_df_large_k, "expected_delta", "expected_delta_var", sqrt_err=True), 1.0),
    ]
    if show_spearman:
        panel_specs.append((axes[4], spearman_df, "correlation", "Spearman ρ", f"(E) k = {k_large}", None, None))

    handles_all, labels_all = [], []
    for ax, df_panel, value_col, y_label, panel_label, exp_map, percent_xmax in panel_specs:
        handles, labels = _draw_panel(
            ax,
            df_panel,
            value_col,
            y_label,
            panel_label,
            exp_map=exp_map,
            percent_xmax=percent_xmax,
        )
        handles_all.extend(handles)
        labels_all.extend(labels)

    for ax in axes[:-1]:
        ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

    seen, handles, labels = set(), [], []
    for handle, label in zip(handles_all, labels_all):
        if label and label not in seen:
            seen.add(label)
            handles.append(handle)
            labels.append(label)

    fig.subplots_adjust(hspace=0.06, bottom=0.12)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.04),
        ncol=min(len(labels), legend_ncols),
        frameon=True,
        edgecolor="black",
        facecolor="white",
    )

    _save_fig_png_pdf(fig, outpath)
    plt.show()


plot_combined_delta_talent_panels(
    delta_df_small_k=delta_df_k1,
    delta_df_mid_k=delta_df,
    talent_df_large_k=talent_rate_df_large_k,
    delta_df_large_k=delta_df_large_k,
    spearman_df=zcp_trd_df_spearman,
    exp_df_small_k=expvals_df_k1,
    exp_df_mid_k=expvals_df,
    exp_df_large_k=expvals_df_large_k,
    outpath=OUT_DIR / "combined_delta_talent_rate_delta.pdf",
    dataset_order=ORDERED_DATASETS,
    k_small=1,
    k_mid=k,
    k_large=K_TO_USE,
    include_spearman=False,
    rs_label="Random search",
)

In [ ]:
"""
plots above as 2x2 grid in accordance with the shepherding feedback to condense the plots and remove the 
spearman panel which reviewer 1 said was not necessary since we already call out that it is not useful.
"""


from matplotlib import gridspec


def plot_combined_delta_talent_panels_2x2(
    *,
    delta_df_small_k,
    delta_df_mid_k,
    talent_df_large_k,
    delta_df_large_k,
    spearman_df=None,
    exp_df_small_k,
    exp_df_mid_k,
    exp_df_large_k,
    outpath,
    dataset_order=None,
    k_small=1,
    k_mid=20,
    k_large=None,
    include_spearman=False,          # flip to True to add panel (E) at the bottom
    figsize_2x2=(16, 10),
    figsize_with_spearman=(16, 13),
    legend_ncols=5,
    rs_label="Random search",
    rs_color="#66CCEE",
    rs_ecolor="#000000",
    rs_err_capsize=4,
):
    def _norm_ds(ds: str) -> str:
        ds = ds.lower()
        return "mobiact" if ds == "mobiactv2" else ds

    def _ds_lbl(ds: str) -> str:
        ds = _norm_ds(ds)
        return ds.capitalize() if ds in {"motionsense", "mobiact", "myogym"} else ds.upper()

    def _exp_map(exp_df, value_col, err_col, *, sqrt_err=False, err_scale=1.0):
        if exp_df is None:
            return {}
        out = {}
        for _, row in exp_df.iterrows():
            ds = _norm_ds(str(row["dataset"]))
            mu = float(row[value_col]) if value_col in row and pd.notna(row[value_col]) else np.nan
            err = float(row[err_col]) if err_col in row and pd.notna(row[err_col]) else np.nan
            if sqrt_err and np.isfinite(err):
                err = float(np.sqrt(err))
            if np.isfinite(err):
                err *= err_scale
            out[ds] = (mu, err)
        return out

    def _style_ax(ax):
        ax.set_facecolor(plt.rcParams["axes.facecolor"])
        ax.grid(True, axis="y", color="white", linewidth=1.2)
        ax.set_axisbelow(True)
        for side in ("top", "right", "left", "bottom"):
            ax.spines[side].set_visible(True)
            ax.spines[side].set_color("black")
            ax.spines[side].set_linewidth(0.9)
        ax.tick_params(axis="both", length=0)

    METRIC_SYMBOL = {
        "delta": "Δ",
        "talent": "τ",
        "correlation": "ρ",
    }

    def _draw_panel(ax, df, value_col, y_label, panel_label, exp_map=None, percent_xmax=None):
        metrics = sorted(df["metric"].unique())
        n_groups = len(metrics) + (1 if exp_map else 0)
        x = np.arange(len(dataset_order))
        width = 0.8 / max(n_groups, 1)
        max_val, min_val = -np.inf, np.inf

        for j, metric in enumerate(metrics):
            offsets = x + (j - n_groups / 2) * width + width / 2
            sub = df[df["metric"] == metric].set_index("dataset")
            vals = [sub.loc[ds, value_col] if ds in sub.index else np.nan for ds in dataset_order]
            ax.bar(
                offsets,
                vals,
                width=width,
                label=metric,
                color=PALETTE10[j % len(PALETTE10)],
                edgecolor="black",
                linewidth=0.8,
            )
            finite_vals = [v for v in vals if np.isfinite(v)]
            if finite_vals:
                max_val = max(max_val, max(finite_vals))
                min_val = min(min_val, min(finite_vals))

        if exp_map:
            j = len(metrics)
            offsets = x + (j - n_groups / 2) * width + width / 2
            vals, yerr = [], []
            for ds in dataset_order:
                mu, err = exp_map.get(_norm_ds(ds), (np.nan, 0.0))
                vals.append(mu)
                yerr.append(err if np.isfinite(err) else 0.0)
                if np.isfinite(mu):
                    err_for_lim = err if np.isfinite(err) else 0.0
                    max_val = max(max_val, mu + err_for_lim)
                    min_val = min(min_val, mu - err_for_lim)

            ax.bar(
                offsets,
                vals,
                width=width,
                label=rs_label,
                color=rs_color,
                edgecolor="black",
                linewidth=0.8,
                yerr=yerr,
                error_kw=dict(
                    elinewidth=1.2,
                    ecolor=rs_ecolor,
                    capsize=rs_err_capsize,
                    capthick=1.2,
                ),
            )

        if percent_xmax is not None:
            ax.yaxis.set_major_formatter(PercentFormatter(xmax=percent_xmax, decimals=0))
        ax.text(
            0.01,
            0.95,
            f"{panel_label} — {METRIC_SYMBOL.get(value_col, y_label)}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=14,
        )
        ax.set_xticks(x)
        ax.set_xticklabels([_ds_lbl(ds) for ds in dataset_order], rotation=0)
        ax.margins(x=0.02)
        _style_ax(ax)

        if value_col == "correlation":
            ax.set_ylim(-0.4, 1.0)
        elif np.isfinite(max_val) and np.isfinite(min_val):
            span = max_val - min_val if max_val > min_val else 1.0
            pad = 0.12 * span
            ax.set_ylim(min_val - 0.2 * pad, max_val + pad)

        return ax.get_legend_handles_labels()

    if dataset_order is None:
        dataset_order = ORDERED_DATASETS
    if k_large is None:
        k_large = K_TO_USE

    show_spearman = include_spearman and spearman_df is not None

    plt.style.use("ggplot")
    if show_spearman:
        fig = plt.figure(figsize=figsize_with_spearman, dpi=200)
        gs = gridspec.GridSpec(3, 2, figure=fig, height_ratios=[1.0, 1.0, 0.75], hspace=0.30, wspace=0.12)
        axes = {
            "A": fig.add_subplot(gs[0, 0]),
            "B": fig.add_subplot(gs[0, 1]),
            "C": fig.add_subplot(gs[1, 0]),
            "D": fig.add_subplot(gs[1, 1]),
            "E": fig.add_subplot(gs[2, :]),
        }
        bottom_margin = 0.14
    else:
        fig, axes_arr = plt.subplots(2, 2, figsize=figsize_2x2, sharex=True, dpi=200)
        axes = {"A": axes_arr[0, 0], "B": axes_arr[0, 1], "C": axes_arr[1, 0], "D": axes_arr[1, 1]}
        bottom_margin = 0.16

    fig.patch.set_facecolor("white")

    panel_specs = [
        (axes["A"], delta_df_small_k, "delta", "Delta from Best Model", f"(A) k = {k_small}", _exp_map(exp_df_small_k, "expected_delta", "expected_delta_var", sqrt_err=True), 1.0),
        (axes["B"], delta_df_mid_k, "delta", "Delta from Best Model", f"(B) k = {k_mid}", _exp_map(exp_df_mid_k, "expected_delta", "expected_delta_var", sqrt_err=True), 1.0),
        (axes["C"], talent_df_large_k, "talent", "Talent Rate", f"(C) k = {k_large}", _exp_map(exp_df_large_k, "expected_talent_rate", "expected_talent_rate_var", err_scale=100.0), 100.0),
        (axes["D"], delta_df_large_k, "delta", "Delta from Best Model", f"(D) k = {k_large}", _exp_map(exp_df_large_k, "expected_delta", "expected_delta_var", sqrt_err=True), 1.0),
    ]
    if show_spearman:
        panel_specs.append(
            (axes["E"], spearman_df, "correlation", "Spearman ρ", f"(E) k = {k_large}", None, None)
        )

    handles_all, labels_all = [], []
    for ax, df_panel, value_col, y_label, panel_label, exp_map, percent_xmax in panel_specs:
        handles, labels = _draw_panel(
            ax,
            df_panel,
            value_col,
            y_label,
            panel_label,
            exp_map=exp_map,
            percent_xmax=percent_xmax,
        )
        handles_all.extend(handles)
        labels_all.extend(labels)

    # hide x labels on top row of the 2x2 block
    for key in ("A", "B"):
        axes[key].tick_params(axis="x", which="both", bottom=False, labelbottom=False)

    seen, handles, labels = set(), [], []
    for handle, label in zip(handles_all, labels_all):
        if label and label not in seen:
            seen.add(label)
            handles.append(handle)
            labels.append(label)

    fig.subplots_adjust(bottom=bottom_margin)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=min(len(labels), legend_ncols),
        frameon=True,
        edgecolor="black",
        facecolor="white",
    )

    _save_fig_png_pdf(fig, outpath)
    plt.show()


# --- generate plot (no Spearman) ---
plot_combined_delta_talent_panels_2x2(
    delta_df_small_k=delta_df_k1,
    delta_df_mid_k=delta_df,
    talent_df_large_k=talent_rate_df_large_k,
    delta_df_large_k=delta_df_large_k,
    spearman_df=zcp_trd_df_spearman,
    exp_df_small_k=expvals_df_k1,
    exp_df_mid_k=expvals_df,
    exp_df_large_k=expvals_df_large_k,
    outpath=OUT_DIR / "combined_delta_talent_rate_delta_2x2.pdf",
    dataset_order=ORDERED_DATASETS,
    k_small=1,
    k_mid=k,
    k_large=K_TO_USE,
    include_spearman=False,   # set True to add panel (E) at the bottom
    rs_label="Random search",
)

In [ ]:
delta_df_k1.groupby("dataset").delta.min()
# delta_df.groupby("dataset").delta.min()
# talent_rate_df_large_k[talent_rate_df_large_k["metric"] == "ensemble"]
# talent_rate_df_large_k.groupby("metric").talent.median()
delta_df_k1

In [ ]:
# Spearman ρ(ensemble, best_test_f1_macro): x grouped by model type; bars = datasets
def _ds_lbl(d: str) -> str:
    d = "mobiact" if d == "mobiactv2" else d
    return d.capitalize() if d.lower() in {"motionsense", "mobiact", "myogym"} else d.upper()

model_types = sorted({
    mt
    for ds in ORDERED_DATASETS
    for mt in pd.read_pickle(DATASET_CONFIGS[ds]["full_results_pkl"])["model_type"].dropna().unique()
})
rows = []
for ds in ORDERED_DATASETS:
    z = _load_and_align(cfg=DATASET_CONFIGS[ds], id_regex=ID_REGEX, keep_run_dir=True)
    fdf = pd.read_pickle(DATASET_CONFIGS[ds]["full_results_pkl"])
    z = z.merge(fdf[["run_dir", "model_type"]], on="run_dir", how="left")
    for mt in model_types:
        sub = z[z["model_type"] == mt]
        if len(sub) < 3:
            rho = np.nan
        else:
            rho = sub[["ensemble", "best_test_f1_macro"]].corr(method="spearman").loc["ensemble", "best_test_f1_macro"]
        rows.append({"dataset": ds, "model_type": mt, "correlation": float(rho)})

corr_df = pd.DataFrame(rows)
x = np.arange(len(model_types))
w = 0.8 / len(ORDERED_DATASETS)
plt.style.use("ggplot")
fig, ax = plt.subplots(figsize=(18, 5), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor(plt.rcParams["axes.facecolor"])
ax.grid(True, axis="y", color="white", linewidth=1.2)
ax.set_axisbelow(True)
for side in ("top", "right", "left", "bottom"):
    ax.spines[side].set_visible(True)
    ax.spines[side].set_color("black")
    ax.spines[side].set_linewidth(0.9)
ax.tick_params(axis="both", length=0)

for j, ds in enumerate(ORDERED_DATASETS):
    off = x + (j - len(ORDERED_DATASETS) / 2) * w + w / 2
    sub = corr_df[corr_df["dataset"] == ds].set_index("model_type")
    vals = [float(sub.loc[mt, "correlation"]) if mt in sub.index and np.isfinite(sub.loc[mt, "correlation"]) else np.nan for mt in model_types]
    ax.bar(off, vals, width=w, label=_ds_lbl(ds), color=PALETTE10[j % len(PALETTE10)], edgecolor="black", linewidth=0.9)

ax.set_ylabel("Spearman ρ")
ax.set_ylim(-0.4, 1.0)
ax.set_xticks(x)
ax.set_xticklabels(model_types, rotation=15, ha="right")
ax.margins(x=0.02)
fig.subplots_adjust(bottom=0.18)
fig.legend(loc="lower center", bbox_to_anchor=(0.5, 0.04), ncol=min(6, len(ORDERED_DATASETS)), frameon=True, edgecolor="black", facecolor="white")

_save_fig_png_pdf(fig, OUT_DIR / "ensemble_spearman_by_model_type.pdf")
plt.show()

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.path_utils import PROJECT_ROOT
PROJECT_ROOT = Path(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.model_testing.dataset_processing_config import DATASET_CONFIGS
from analysis.model_testing.zero_cost_metric_analysis import ZERO_COST_METRICS, METRIC_DIRECTION

DATASETS = ["myogym", "hhar", "mobiactv2", "motionsense", "pamap2", "rwhar"]
FLOPS_SELECTIONS = [
    ("top 1", lambda sub: 1),
    ("top 10", lambda sub: 10),
    ("top 10%", lambda sub: int(np.ceil(len(sub) * 0.10))),
    ("full-scale training", lambda sub: len(sub)),
]
FLOPS_COLORS = PALETTE10[:len(FLOPS_SELECTIONS)]

OUT_DIR = PROJECT_ROOT / "analysis/figures/current/talented_models/may_2026_rerun"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def flip_zcp_and_ensemble(zcp_df):
    zc = zcp_df.copy()
    valid_metrics = [m for m in ZERO_COST_METRICS if m in zc.columns]
    for m, sign in METRIC_DIRECTION.items():
        if sign == -1 and m in zc.columns:
            zc[m] = zc[m] * -1
    zc["ensemble"] = zc[valid_metrics].rank(pct=True).mean(axis=1)
    return zc

rows = []
for name in DATASETS:
    cfg = DATASET_CONFIGS[name]
    flops = pd.read_pickle(cfg["model_flops_pkl"])[["run_dir", "flops"]]
    full = pd.read_pickle(cfg["full_results_pkl"])[["run_dir", "best_val_f1_macro"]]
    zcp = flip_zcp_and_ensemble(pd.read_pickle(cfg["zero_cost_pkl"]))[["run_dir", "ensemble"]]

    df = flops.merge(zcp, on="run_dir", how="inner").merge(full, on="run_dir", how="inner")
    df["dataset"] = name
    rows.append(df)

all_models_df = pd.concat(rows, ignore_index=True)

totals = np.zeros((len(DATASETS), len(FLOPS_SELECTIONS)))
for i, ds in enumerate(DATASETS):
    sub = all_models_df[all_models_df["dataset"] == ds].sort_values("ensemble", ascending=False)
    for j, (_, k_fn) in enumerate(FLOPS_SELECTIONS):
        k = min(max(1, k_fn(sub)), len(sub))
        totals[i, j] = sub.head(k)["flops"].sum()

x = np.arange(len(DATASETS))
w = 0.8 / len(FLOPS_SELECTIONS)

plt.style.use("ggplot")
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
})

fig, ax = plt.subplots(figsize=(9, 4), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor(plt.rcParams["axes.facecolor"])
ax.grid(True, axis="y", color="white", linewidth=1.2)
ax.set_axisbelow(True)
for side in ("top", "right", "left", "bottom"):
    ax.spines[side].set_visible(True)
    ax.spines[side].set_color("black")
    ax.spines[side].set_linewidth(0.9)
ax.tick_params(axis="both", length=0)

for j, (label, _) in enumerate(FLOPS_SELECTIONS):
    ax.bar(
        x + (j - len(FLOPS_SELECTIONS) / 2) * w + w / 2,
        totals[:, j],
        width=w,
        label=label,
        color=PALETTE10[j % len(PALETTE10)],
        edgecolor="black",
        linewidth=0.8,
    )

ax.set_ylabel("Total FLOPs")
ax.set_xticks(x)
ax.set_xticklabels(["Myogym", "HHAR", "Mobiact", "Motionsense", "PAMAP2", "RWHAR"])
ax.set_yscale("log")
ax.margins(x=0.02)

handles, labels = ax.get_legend_handles_labels()
fig.subplots_adjust(bottom=0.24)
fig.legend(
    handles,
    labels, 
    loc="lower center",
    bbox_to_anchor=(0.5, 0.04),
    ncol=min(len(labels), len(FLOPS_SELECTIONS)),
    frameon=True,
    edgecolor="black",
    facecolor="white",
)

fig.savefig(OUT_DIR / "total_flops_by_zcp_rank_may_2026.pdf", bbox_inches="tight")
fig.savefig(OUT_DIR / "total_flops_by_zcp_rank_may_2026.png", dpi=300, bbox_inches="tight")
plt.show()

all_models_df.head()

In [ ]:
dataset = "motionsense"
mdf = pd.read_pickle(DATASET_CONFIGS[dataset]['zero_cost_pkl'])
mdf.head()

In [ ]:
# CNN/RNN-only version of the combined delta/talent/Spearman plot.
# Reuses compute_talent_stats() and plot_combined_delta_talent_panels() from above.
from IPython.display import display
import inspect

NO_TRANSFORMER_MODEL_TYPES = ("CNN", "RNN")

if "model_types" not in inspect.signature(compute_talent_stats).parameters:
    raise RuntimeError("Please rerun the helper cell that defines compute_talent_stats() before this cell.")

no_transformer_talent_k1, no_transformer_delta_k1, no_transformer_exp_k1 = compute_talent_stats(
    use_abs_zcp=False,
    datasets=ORDERED_DATASETS,
    k=1,
    model_types=NO_TRANSFORMER_MODEL_TYPES,
)
no_transformer_talent_k10, no_transformer_delta_k10, no_transformer_exp_k10 = compute_talent_stats(
    use_abs_zcp=False,
    datasets=ORDERED_DATASETS,
    k=10,
    model_types=NO_TRANSFORMER_MODEL_TYPES,
)
no_transformer_talent_k150, no_transformer_delta_k150, no_transformer_exp_k150 = compute_talent_stats(
    use_abs_zcp=False,
    datasets=ORDERED_DATASETS,
    k=150,
    model_types=NO_TRANSFORMER_MODEL_TYPES,
)

for df, k_val in [
    (no_transformer_delta_k1, 1),
    (no_transformer_delta_k10, 10),
    (no_transformer_delta_k150, 150),
]:
    df["k"] = k_val
for df, k_val in [
    (no_transformer_talent_k1, 1),
    (no_transformer_talent_k10, 10),
    (no_transformer_talent_k150, 150),
    (no_transformer_exp_k1, 1),
    (no_transformer_exp_k10, 10),
    (no_transformer_exp_k150, 150),
]:
    df["k"] = k_val

# Spearman panel for the same CNN/RNN-only model pool.
no_transformer_spearman_rows = []
for ds in ORDERED_DATASETS:
    zcp = _load_and_align(cfg=DATASET_CONFIGS[ds], id_regex=ID_REGEX, keep_run_dir=True)
    fdf = pd.read_pickle(DATASET_CONFIGS[ds]["full_results_pkl"])[["run_dir", "model_type"]]
    zcp = zcp.merge(fdf, on="run_dir", how="inner")
    zcp = zcp[zcp["model_type"].isin(NO_TRANSFORMER_MODEL_TYPES)].copy()
    zcp["ensemble"] = zcp[ZC_METRICS].rank(pct=True).mean(axis=1)
    for metric in ZC_METRICS + ["ensemble", "initial_val_f1_macro"]:
        if metric in zcp.columns and "best_test_f1_macro" in zcp.columns:
            rho = zcp[[metric, "best_test_f1_macro"]].corr(method="spearman").loc[metric, "best_test_f1_macro"]
            no_transformer_spearman_rows.append({"dataset": ds, "metric": metric, "correlation": float(rho)})
no_transformer_spearman_df = pd.DataFrame(no_transformer_spearman_rows)

# Same ZCP order used by the original plotting helper: sorted metric labels.
no_transformer_metric_order = sorted(no_transformer_delta_k1["metric"].unique())

no_transformer_delta_table = (
    pd.concat([no_transformer_delta_k1, no_transformer_delta_k10, no_transformer_delta_k150])
    .assign(delta_pct=lambda d: d["delta"] * 100)
    .pivot_table(index=["dataset", "metric"], columns="k", values="delta_pct")
    .rename(columns={1: "delta@1", 10: "delta@10", 150: "delta@150"})
    .reindex(
        pd.MultiIndex.from_product(
            [ORDERED_DATASETS, no_transformer_metric_order],
            names=["dataset", "metric"],
        )
    )
)
no_transformer_talent_k150_table = (
    no_transformer_talent_k150
    .pivot_table(index="dataset", columns="metric", values="talent")
    .reindex(index=ORDERED_DATASETS, columns=no_transformer_metric_order)
)

# print("No TRANSFORMER: delta@1, delta@10, delta@150 (percentage points)")
# display(no_transformer_delta_table)
# print("No TRANSFORMER: talent rate at k=150 (%)")
# display(no_transformer_talent_k150_table)

plot_combined_delta_talent_panels(
    delta_df_small_k=no_transformer_delta_k1,
    delta_df_mid_k=no_transformer_delta_k10,
    talent_df_large_k=no_transformer_talent_k150,
    delta_df_large_k=no_transformer_delta_k150,
    spearman_df=no_transformer_spearman_df,
    exp_df_small_k=no_transformer_exp_k1,
    exp_df_mid_k=no_transformer_exp_k10,
    exp_df_large_k=no_transformer_exp_k150,
    outpath=OUT_DIR / "combined_delta_talent_rate_delta_cnn_rnn_k1_k10_k150.pdf",
    dataset_order=ORDERED_DATASETS,
    k_small=1,
    k_mid=10,
    k_large=150,
    figsize=(18, 16),
    legend_ncols=5,
    rs_label="Random search",
)

In [ ]:
# Best full-training validation model and #1 ZCP ensemble-ranked model.
# Reuses the notebook's DATASET_CONFIGS, ORDERED_DATASETS, ZC_METRICS, ID_REGEX,
# _load_and_align, and extract_model_id_from_run_dir.
import json
from pathlib import Path

from data.dataset_registry import dataset_registry
from models.init_fns import INIT_FN_MAP
from models.model_generator_registry import model_generator_registry
from utils.utils import set_seed, set_seed_deterministic


def _load_trial_params(cfg, run_dir):
    param_path = Path(cfg["results_dir"]) / run_dir / "params.json"
    if not param_path.exists():
        matches = list(Path(cfg["results_dir"]).glob(f"{run_dir}*/params.json"))
        if not matches:
            raise FileNotFoundError(param_path)
        param_path = matches[0]
    return json.loads(param_path.read_text())


def _model_from_row(dataset, row, params):
    dataset_cls = dataset_registry.get_dataset(dataset)
    init_fn = INIT_FN_MAP[params["init_name"]]

    set_seed(int(params["arch_seed"]))
    set_seed_deterministic(int(params["init_seed"]))

    return model_generator_registry.get_generator(row["model_type"])(
        int(dataset_cls.config["input_size"]),
        int(dataset_cls.config["num_classes"]),
        init_fn=init_fn,
    ).generate_model(fixed_arch_config=params["arch_config"])


def _package_selected_architecture(dataset, selection, row):
    cfg = DATASET_CONFIGS[dataset]
    params = _load_trial_params(cfg, row["run_dir"])
    model = _model_from_row(dataset, row, params)

    return {
        "dataset": dataset,
        "selection": selection,
        "run_dir": row["run_dir"],
        "model_id": extract_model_id_from_run_dir(pd.Series([row["run_dir"]])).iloc[0],
        "model_type": row["model_type"],
        "best_val_f1_macro": float(row["best_val_f1_macro"]),
        "best_test_f1_macro": float(row["best_test_f1_macro"]),
        "ensemble": float(row["ensemble"]) if "ensemble" in row and pd.notna(row["ensemble"]) else None,
        "num_params": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        "arch_config": params["arch_config"],
        "model_object": model,
    }


selected_architectures = {}

for dataset in ORDERED_DATASETS:
    cfg = DATASET_CONFIGS[dataset]

    full_results = pd.read_pickle(cfg["full_results_pkl"]).dropna(
        subset=["best_val_f1_macro", "best_test_f1_macro", "model_type"]
    ).copy()
    full_results["model_id"] = extract_model_id_from_run_dir(full_results["run_dir"])

    zcp = _load_and_align(cfg=cfg, id_regex=ID_REGEX, keep_run_dir=True, use_abs=False)
    zcp = zcp.merge(
        full_results[["run_dir", "best_val_f1_macro", "model_id", "model_type"]],
        on="run_dir",
        how="inner",
    ).dropna(subset=["best_val_f1_macro", "best_test_f1_macro", "model_type", "ensemble"])

    full_best = full_results.sort_values("best_val_f1_macro", ascending=False).iloc[0]
    zcp_best = zcp.sort_values("ensemble", ascending=False).iloc[0]

    selected_architectures[(dataset, "full_best_val")] = _package_selected_architecture(
        dataset, "full_best_val", full_best
    )
    selected_architectures[(dataset, "zcp_ensemble_rank_1")] = _package_selected_architecture(
        dataset, "zcp_ensemble_rank_1", zcp_best
    )

selected_architecture_summary = pd.DataFrame([
    {k: v for k, v in obj.items() if k not in {"arch_config", "model_object"}}
    for obj in selected_architectures.values()
])

display(selected_architecture_summary)

param_gap_by_dataset = (
    selected_architecture_summary
    .pivot(index="dataset", columns="selection", values="num_params")
    .assign(
        param_diff=lambda df: df["zcp_ensemble_rank_1"] - df["full_best_val"],
        abs_param_diff=lambda df: df["param_diff"].abs(),
        param_quotient=lambda df: df[["full_best_val", "zcp_ensemble_rank_1"]].max(axis=1)
        / df[["full_best_val", "zcp_ensemble_rank_1"]].min(axis=1),
    )
    .sort_values("param_quotient", ascending=False)
)

display(param_gap_by_dataset)

starkest_dataset = param_gap_by_dataset.index[0]
starkest_full = selected_architectures[(starkest_dataset, "full_best_val")]
starkest_zcp = selected_architectures[(starkest_dataset, "zcp_ensemble_rank_1")]

starkest_param_ratio = param_gap_by_dataset.loc[starkest_dataset, "param_quotient"]

print(f"Starkest parameter-count ratio: {starkest_dataset}")
print(
    f"full_best_val params = {starkest_full['num_params']:,}; "
    f"zcp_ensemble_rank_1 params = {starkest_zcp['num_params']:,}; "
    f"absolute difference = {abs(starkest_zcp['num_params'] - starkest_full['num_params']):,}; "
    f"quotient = {starkest_param_ratio:.2f}x"
)

print("\nFull best-val selected architecture config:")
print(starkest_full["arch_config"])
print("\nFull best-val model object:")
print(starkest_full["model_object"])

print("\n#1 ZCP ensemble-ranked architecture config:")
print(starkest_zcp["arch_config"])
print("\n#1 ZCP ensemble-ranked model object:")
print(starkest_zcp["model_object"])

mobiactv2_full = selected_architectures[("mobiactv2", "full_best_val")]
mobiactv2_zcp = selected_architectures[("mobiactv2", "zcp_ensemble_rank_1")]

print("\n\nMOBIACTV2 full_best_val architecture config:")
print(mobiactv2_full["arch_config"])
print("\nMOBIACTV2 full_best_val model object:")
print(mobiactv2_full["model_object"])

print("\nMOBIACTV2 zcp_ensemble_rank_1 architecture config:")
print(mobiactv2_zcp["arch_config"])
print("\nMOBIACTV2 zcp_ensemble_rank_1 model object:")
print(mobiactv2_zcp["model_object"])

# Examples:
# selected_architectures[("motionsense", "full_best_val")]["model_object"]
# selected_architectures[("motionsense", "full_best_val")]["arch_config"]

In [ ]:
# Architecture summaries split by model type.
# Run the previous cell first so _package_selected_architecture is defined.
from IPython.display import display

ARCHITECTURE_ANALYSIS_GROUPS = {
    "CNN": ("CNN",),
    "RNN": ("RNN",),
    "TRANSFORMER": ("TRANSFORMER",),
    "ALL": None,
}


def build_selected_architecture_summary_for_model_types(model_types=None, *, label="ALL"):
    selected = {}
    for dataset in ORDERED_DATASETS:
        cfg = DATASET_CONFIGS[dataset]
        full_results = pd.read_pickle(cfg["full_results_pkl"])
        zcp = _load_and_align(cfg=cfg, id_regex=ID_REGEX, keep_run_dir=True)

        full_cols = ["run_dir", "best_val_f1_macro", "best_test_f1_macro", "model_type"]
        full_results = full_results[full_cols].dropna(
            subset=["best_val_f1_macro", "best_test_f1_macro", "model_type"]
        )
        zcp = zcp.merge(
            full_results[["run_dir", "best_val_f1_macro", "model_type"]],
            on="run_dir",
            how="inner",
        )

        if model_types is not None:
            full_results = full_results[full_results["model_type"].isin(model_types)].copy()
            zcp = zcp[zcp["model_type"].isin(model_types)].copy()

        if full_results.empty or zcp.empty:
            raise ValueError(f"No rows available for dataset={dataset}, model_types={model_types}")

        zcp["ensemble"] = zcp[ZC_METRICS].rank(pct=True).mean(axis=1)
        full_best = full_results.sort_values("best_val_f1_macro", ascending=False).iloc[0]
        zcp_best = zcp.sort_values("ensemble", ascending=False).iloc[0]

        selected[(dataset, "full_best_val")] = _package_selected_architecture(
            dataset, "full_best_val", full_best
        )
        selected[(dataset, "zcp_ensemble_rank_1")] = _package_selected_architecture(
            dataset, "zcp_ensemble_rank_1", zcp_best
        )

    summary = pd.DataFrame(selected.values())[
        [
            "dataset",
            "selection",
            "run_dir",
            "model_id",
            "model_type",
            "best_val_f1_macro",
            "best_test_f1_macro",
            "ensemble",
            "num_params",
        ]
    ]
    summary.insert(0, "analysis_group", label)
    return summary


selected_architecture_summaries_by_model_type = {
    label: build_selected_architecture_summary_for_model_types(model_types, label=label)
    for label, model_types in ARCHITECTURE_ANALYSIS_GROUPS.items()
}

selected_architecture_summary_all_groups = pd.concat(
    selected_architecture_summaries_by_model_type.values(), ignore_index=True
)

for label, summary in selected_architecture_summaries_by_model_type.items():
    print(f"{label} selected architectures")
    display(summary)

selected_architecture_summary_all_groups

In [ ]:
# Evidence for the model-family architectural-bias paragraph.
# Run the previous cell first so selected_architecture_summary_all_groups exists.
from IPython.display import display


def infer_arch_depth(arch_config):
    """Return a simple comparable layer/block count for the saved architecture config."""
    if isinstance(arch_config, list):
        return len(arch_config)
    if isinstance(arch_config, dict):
        for key in ["num_layers", "n_layers", "num_blocks", "n_blocks", "num_encoder_layers"]:
            if key in arch_config:
                return arch_config[key]
        list_lengths = [len(v) for v in arch_config.values() if isinstance(v, list)]
        if list_lengths:
            return max(list_lengths)
    return np.nan


def add_arch_depth(summary_df):
    rows = []
    for row in summary_df.itertuples(index=False):
        params = _load_trial_params(DATASET_CONFIGS[row.dataset], row.run_dir)
        rows.append({
            **row._asdict(),
            "arch_depth": infer_arch_depth(params["arch_config"]),
        })
    return pd.DataFrame(rows)


architecture_evidence_long = add_arch_depth(selected_architecture_summary_all_groups)

architecture_evidence = (
    architecture_evidence_long
    .pivot_table(
        index=["analysis_group", "dataset"],
        columns="selection",
        values=["model_type", "num_params", "arch_depth", "best_val_f1_macro", "best_test_f1_macro", "ensemble"],
        aggfunc="first",
    )
)
architecture_evidence.columns = [f"{field}_{selection}" for field, selection in architecture_evidence.columns]
architecture_evidence = architecture_evidence.reset_index()
architecture_evidence["zcp_depth_delta"] = (
    architecture_evidence["arch_depth_zcp_ensemble_rank_1"]
    - architecture_evidence["arch_depth_full_best_val"]
)
architecture_evidence["zcp_param_ratio"] = (
    architecture_evidence["num_params_zcp_ensemble_rank_1"]
    / architecture_evidence["num_params_full_best_val"]
)
architecture_evidence["zcp_same_model_type_as_full_best"] = (
    architecture_evidence["model_type_zcp_ensemble_rank_1"]
    == architecture_evidence["model_type_full_best_val"]
)
architecture_evidence["zcp_depth_relation"] = np.select(
    [
        architecture_evidence["zcp_depth_delta"] < 0,
        architecture_evidence["zcp_depth_delta"] == 0,
        architecture_evidence["zcp_depth_delta"] > 0,
    ],
    ["shallower", "equal", "deeper"],
    default="unknown",
)

architecture_evidence_summary = (
    architecture_evidence
    .groupby("analysis_group")
    .agg(
        n_datasets=("dataset", "count"),
        zcp_shallower=("zcp_depth_relation", lambda s: int((s == "shallower").sum())),
        zcp_equal_depth=("zcp_depth_relation", lambda s: int((s == "equal").sum())),
        zcp_deeper=("zcp_depth_relation", lambda s: int((s == "deeper").sum())),
        median_zcp_depth_delta=("zcp_depth_delta", "median"),
        median_zcp_param_ratio=("zcp_param_ratio", "median"),
        zcp_same_model_type_as_full_best=("zcp_same_model_type_as_full_best", "sum"),
    )
)

pooled_zcp_model_type_counts = (
    architecture_evidence
    .query("analysis_group == 'ALL'")
    ["model_type_zcp_ensemble_rank_1"]
    .value_counts()
    .rename_axis("zcp_selected_model_type")
    .reset_index(name="n_datasets")
)

print("Per-dataset evidence for full-training vs. #1 ZCP ensemble selections")
display(
    architecture_evidence[
        [
            "analysis_group",
            "dataset",
            "model_type_full_best_val",
            "model_type_zcp_ensemble_rank_1",
            "arch_depth_full_best_val",
            "arch_depth_zcp_ensemble_rank_1",
            "zcp_depth_relation",
            "zcp_depth_delta",
            "num_params_full_best_val",
            "num_params_zcp_ensemble_rank_1",
            "zcp_param_ratio",
        ]
    ]
)

print("Summary by analysis group")
display(architecture_evidence_summary)

print("Pooled ALL setting: #1 ZCP-selected model type counts")
display(pooled_zcp_model_type_counts)

cnn = architecture_evidence_summary.loc["CNN"]
rnn = architecture_evidence_summary.loc["RNN"]
transformer = architecture_evidence_summary.loc["TRANSFORMER"]
all_group = architecture_evidence_summary.loc["ALL"]
all_transformer_count = int(
    pooled_zcp_model_type_counts
    .set_index("zcp_selected_model_type")
    .get("n_datasets", pd.Series(dtype=int))
    .get("TRANSFORMER", 0)
)

print(
    f"CNN-only: ZCP is shallower in {int(cnn.zcp_shallower)}/{int(cnn.n_datasets)} datasets, "
    f"equal in {int(cnn.zcp_equal_depth)}/{int(cnn.n_datasets)}, and deeper in {int(cnn.zcp_deeper)}/{int(cnn.n_datasets)}."
)
print(
    f"RNN-only: ZCP is shallower in {int(rnn.zcp_shallower)}/{int(rnn.n_datasets)} datasets, "
    f"equal in {int(rnn.zcp_equal_depth)}/{int(rnn.n_datasets)}, and deeper in {int(rnn.zcp_deeper)}/{int(rnn.n_datasets)}."
)
print(
    f"Transformer-only: ZCP is shallower in {int(transformer.zcp_shallower)}/{int(transformer.n_datasets)} datasets, "
    f"equal in {int(transformer.zcp_equal_depth)}/{int(transformer.n_datasets)}, and deeper in {int(transformer.zcp_deeper)}/{int(transformer.n_datasets)}."
)
print(
    f"All-model pooled setting: the #1 ZCP ensemble selection is TRANSFORMER in "
    f"{all_transformer_count}/{int(all_group.n_datasets)} datasets."
)

In [ ]:
dataset = "mobiactv2"
cfg = DATASET_CONFIGS[dataset]

full_results = pd.read_pickle(cfg["full_results_pkl"]).copy()
full_results["model_id"] = full_results["run_dir"].str.extract(ID_REGEX)[0]

zcp = _load_and_align(cfg=cfg, id_regex=ID_REGEX, keep_run_dir=True, use_abs=False)
zcp_aligned = zcp.merge(
    full_results[["run_dir", "best_val_f1_macro", "model_type", "model_id"]],
    on="run_dir",
    how="inner",
)

print(
    f"{dataset}: {len(zcp_aligned)} aligned rows "
    f"({len(full_results)} full-training rows, {len(zcp)} zero-cost rows)"
)
display(zcp_aligned.head())

In [ ]:
import seaborn as sns
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Standalone: skips the 1,000-draw random baseline, so this cell can run by itself.
cwd = Path.cwd()
project_root = None
current = cwd
for _ in range(5):
    if (current / "train.py").exists():
        project_root = current
        break
    current = current.parent
if project_root and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.path_utils import setup_path
setup_path()

from analysis.talented_models_v3 import ZC_METRICS, ORDERED_DATASETS, OUT_DIR as _DEFAULT_OUT_DIR
from analysis.model_testing.dataset_processing_config import DATASET_CONFIGS
from analysis.model_testing.zero_cost_metric_analysis import ID_REGEX, _load_and_align

OUT_DIR = _DEFAULT_OUT_DIR / "may_2026_rerun"
MODEL_TYPE_FILTER = None
K_TOP_10_PERCENT = 200  # 10% of the 2,000 sampled architectures

_probe = Path(DATASET_CONFIGS["myogym"]["full_results_pkl"])
if not _probe.exists():
    raise FileNotFoundError(
        f"{_probe} is missing. Set MGT_PAPER_RESULTS_ROOT to the run's results "
        "directory before starting the kernel."
    )

DATASET_LABELS = {
    "myogym": "Myogym",
    "hhar": "HHAR",
    "mobiactv2": "Mobiact",
    "motionsense": "Motionsense",
    "pamap2": "PAMAP2",
    "rwhar": "RWHAR",
}
SELECTION_ORDER = ["Top 1", "Top 10", "Top 10%"]
K_BY_SELECTION = {"Top 1": 1, "Top 10": 10, "Top 10%": K_TOP_10_PERCENT}
PROXY_METRICS = ["initial_val_f1_macro"] + list(ZC_METRICS) + ["ensemble"]


def _save_fig_png_pdf(fig, outpath):
    outpath = Path(outpath)
    outpath.parent.mkdir(parents=True, exist_ok=True)
    stem = outpath.stem
    if MODEL_TYPE_FILTER is not None:
        stem = f"{stem}_model_filter={MODEL_TYPE_FILTER}"
    base = outpath.with_name(stem)
    for suf in (".pdf", ".png"):
        p = base.with_suffix(suf)
        fig.savefig(p, dpi=300, bbox_inches="tight")
        print(f"[plot saved → {p}]")


def _models_with_flops(dataset):
    cfg = DATASET_CONFIGS[dataset]
    zcp = _load_and_align(cfg=cfg, id_regex=ID_REGEX, keep_run_dir=True)
    fdf = pd.read_pickle(cfg["full_results_pkl"]).drop("initial_val_f1_macro", axis=1, errors="ignore")
    zcp = zcp.merge(fdf[["run_dir", "best_val_f1_macro", "model_type"]], on="run_dir")
    if MODEL_TYPE_FILTER is not None:
        zcp = zcp[zcp["model_type"] == MODEL_TYPE_FILTER]
    zcp["ensemble"] = zcp[list(ZC_METRICS)].rank(pct=True).mean(axis=1)
    flops = pd.read_pickle(cfg["model_flops_pkl"])[["run_dir", "flops"]]
    return zcp.merge(flops, on="run_dir", how="left")


best_rows = []
savings_rows = []
for dataset in ORDERED_DATASETS:
    models = _models_with_flops(dataset)
    full_flops = float(models["flops"].sum())
    for selection, k in K_BY_SELECTION.items():
        k_eff = min(k, len(models))
        top_by_val = models.sort_values("best_val_f1_macro", ascending=False).head(k_eff)
        top_test = float(top_by_val["best_test_f1_macro"].iloc[0])
        per_metric = []
        for metric in PROXY_METRICS:
            ranked = models.sort_values(metric, ascending=False).head(k_eff)
            chosen = ranked.sort_values("best_val_f1_macro", ascending=False).iloc[0]
            per_metric.append({
                "dataset": dataset,
                "selection": selection,
                "metric": metric,
                "delta": top_test - float(chosen["best_test_f1_macro"]),
                "selected_flops": float(ranked["flops"].sum()),
            })
        best = min(per_metric, key=lambda row: row["delta"])
        best_rows.append(best)
        savings_rows.append({
            "dataset": dataset,
            "selection": selection,
            "selected_flops": best["selected_flops"],
            "full_flops": full_flops,
            "savings_pct": 100.0 * (1.0 - best["selected_flops"] / full_flops),
        })

best_proxy_df = pd.DataFrame(best_rows)
best_proxy_df["delta_pct"] = best_proxy_df["delta"] * 100
best_proxy_df["dataset_label"] = best_proxy_df["dataset"].map(DATASET_LABELS)
savings_df = pd.DataFrame(savings_rows)
savings_df["dataset_label"] = savings_df["dataset"].map(DATASET_LABELS)


chart_df = best_proxy_df.merge(
    savings_df.drop(columns=["selected_flops"]),
    on=["dataset", "selection", "dataset_label"],
)
dataset_order = [DATASET_LABELS[d] for d in ORDERED_DATASETS]
view = chart_df.copy()
view["dataset_label"] = pd.Categorical(view["dataset_label"], dataset_order, ordered=True)
view["selection"] = pd.Categorical(view["selection"], SELECTION_ORDER, ordered=True)
view["delta_pct"] = view["delta_pct"].map(lambda v: f"{v:+.2f}")
view["savings_pct"] = view["savings_pct"].map(lambda v: f"{v:.2f}%")
view["selected_flops"] = view["selected_flops"].map(lambda v: f"{v:.3e}")
display(
    view.sort_values(["selection", "dataset_label"])[
        ["dataset_label", "selection", "metric", "delta_pct", "savings_pct", "selected_flops"]
    ]
    .rename(columns={
        "dataset_label": "Dataset",
        "selection": "Budget",
        "metric": "Best proxy",
        "delta_pct": "Gap (pp)",
        "savings_pct": "FLOP savings",
        "selected_flops": "Selected FLOPs",
    })
    .reset_index(drop=True)
)

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.15)
palette = sns.color_palette("colorblind", n_colors=len(SELECTION_ORDER))

fig, ax = plt.subplots(figsize=(12, 5.8))
fig.patch.set_facecolor("white")
sns.barplot(
    data=chart_df,
    x="dataset_label",
    y="delta_pct",
    hue="selection",
    order=dataset_order,
    hue_order=SELECTION_ORDER,
    palette=palette,
    edgecolor="black",
    linewidth=0.6,
    ax=ax,
)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xlabel("")
ax.set_ylabel("Gap from best model (percentage points)")
ax.set_title("Best Proxy on Each Dataset")
for container in ax.containers:
    ax.bar_label(container, fmt="%+.1f", padding=3, fontsize=9)
y_pad = 2.5
ax.set_ylim(min(0, chart_df["delta_pct"].min()) - y_pad, chart_df["delta_pct"].max() + y_pad + 1.5)
sns.move_legend(ax, "lower center", bbox_to_anchor=(0.5, -0.18), ncol=3, title=None, frameon=True)
fig.tight_layout()
fig.subplots_adjust(bottom=0.18)
_save_fig_png_pdf(fig, OUT_DIR / "best_proxy_gap_by_budget.pdf")
plt.show()

fig, ax = plt.subplots(figsize=(12, 5.8))
fig.patch.set_facecolor("white")
sns.barplot(
    data=chart_df,
    x="dataset_label",
    y="savings_pct",
    hue="selection",
    order=dataset_order,
    hue_order=SELECTION_ORDER,
    palette=palette,
    edgecolor="black",
    linewidth=0.6,
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("FLOPs saved vs. exhaustive training (%)")
ax.set_title("Compute savings from training the best proxy's top models")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3, fontsize=9)
ax.set_ylim(0, 118)
sns.move_legend(ax, "lower center", bbox_to_anchor=(0.5, -0.18), ncol=3, title=None, frameon=True)
fig.tight_layout()
fig.subplots_adjust(bottom=0.18)
_save_fig_png_pdf(fig, OUT_DIR / "best_proxy_compute_savings.pdf")
plt.show()


def _extract_model_id(run_dir_series):
    model_id = run_dir_series.str.extract(r"(arch_seed=\d+,combo_idx=\d+)")
    if model_id[0].isna().all():
        model_id = run_dir_series.str.extract(r"(arch_seed=\d+,init_name=\w+,init_seed=\d+)")
    if model_id[0].isna().all():
        model_id = run_dir_series.str.extract(r"(train_tune_\w+_\d+_\d+)")
    return model_id[0]


# Talent rate: share of the proxy's top 10% that is also in the top 10% by validation F1.
talent_rows = []
for dataset in ORDERED_DATASETS:
    models = _models_with_flops(dataset)
    models["model_id"] = _extract_model_id(models["run_dir"])
    k = min(K_TOP_10_PERCENT, len(models))
    top_actual = set(models.sort_values("best_val_f1_macro", ascending=False).head(k)["model_id"])
    best = None
    for metric in PROXY_METRICS:
        top_pred = set(models.sort_values(metric, ascending=False).head(k)["model_id"])
        rate = 100.0 * len(top_pred & top_actual) / len(top_actual)
        if best is None or rate > best["talent_rate_pct"]:
            best = {"dataset": dataset, "metric": metric, "talent_rate_pct": rate, "k": k}
    best["dataset_label"] = DATASET_LABELS[dataset]
    talent_rows.append(best)
talent_df = pd.DataFrame(talent_rows)
talent_df["dataset_label"] = pd.Categorical(talent_df["dataset_label"], dataset_order, ordered=True)
talent_df = talent_df.sort_values("dataset_label")
display(
    talent_df[["dataset_label", "metric", "talent_rate_pct", "k"]]
    .rename(columns={
        "dataset_label": "Dataset",
        "metric": "Best proxy",
        "talent_rate_pct": "Talent rate (%)",
        "k": "k",
    })
    .reset_index(drop=True)
)
print(
    f"talent rate range: {talent_df['talent_rate_pct'].min():.1f}% – {talent_df['talent_rate_pct'].max():.1f}%"
)

fig, ax = plt.subplots(figsize=(10, 5.6))
fig.patch.set_facecolor("white")
sns.barplot(
    data=talent_df,
    x="dataset_label",
    y="talent_rate_pct",
    order=dataset_order,
    color=sns.color_palette("colorblind")[0],
    edgecolor="black",
    linewidth=0.6,
    ax=ax,
)
ax.axhline(10, color="0.35", linestyle="--", linewidth=1.1, label="Random search (10%)")
ax.set_xlabel("")
ax.set_ylabel("Talent rate (%)")
ax.set_title("Best Proxy Talent Rate for Top 10% of Models")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3, fontsize=11)
ax.set_ylim(0, 62)
ax.legend(loc="upper right", frameon=True)
fig.tight_layout()
_save_fig_png_pdf(fig, OUT_DIR / "best_proxy_talent_rate.pdf")
plt.show()
